In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X,K):
    '''计算二维互相关运算'''
    '''
    Y是输出矩阵
    X是输入矩阵
    K是参数矩阵
    h,w 是K的行和列
    '''
    h,w=K.shape
    Y=torch.zeros((X.shape[0]-h+1,X.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(X[i:i+h,j:j+w]*K).sum()
    return Y



验证二位互相关运算的输出

In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X,K)

tensor([[19., 25.],
        [37., 43.]])

实现二维卷积层

In [ ]:
class Conv2D(nn.Module):
    def __init__(self,kernel_size): 
        '''kernel_size是参数的形状的意思'''
        super().__init__()
        self.weight=nn.Parameter(torch.rand(kernel_size))
        self.bias=nn.Parameter(torch.zeros(1))
    
    def forward(self,x):
        return corr2d(x,self.weight)+self.bias

现在来做一个卷积层的简单应用：检测图片中不同颜色的边缘

In [4]:
X=torch.ones((6,8))
X[:,2:6]=0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [5]:
K =torch.tensor([[1.0,-1.0]])
Y=corr2d(X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

1->白到黑的边沿，-1->黑到白的边沿

学习由X和Y生成的卷积核

In [11]:
conv2d=nn.Conv2d(1,1,kernel_size=(1,2),bias=False)

X=X.reshape((1,1,6,8))
Y=Y.reshape((1,1,6,7))

for i in range(100):
    Y_hat=conv2d(X)
    l=(Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:]-=(3e-2)* conv2d.weight.grad
    if (i+1)%10 == 0:
        print(f'batch: {i+1},loss: {l.sum():.3f}')
        

batch: 10,loss: 0.109
batch: 20,loss: 0.001
batch: 30,loss: 0.000
batch: 40,loss: 0.000
batch: 50,loss: 0.000
batch: 60,loss: 0.000
batch: 70,loss: 0.000
batch: 80,loss: 0.000
batch: 90,loss: 0.000
batch: 100,loss: 0.000


In [12]:
conv2d.weight.data.reshape((1,2))

tensor([[ 1.0000, -1.0000]])